---
## 0. Setup and load

In [163]:
import pandas as pd
from pandas import DataFrame,Series
import numpy as np
df = pd.read_csv(r"C:\Users\ASUS\Desktop\Data Analysis\Pandas\datasets\Airbnb_Open_Data.csv")
df.head(5)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_28456\524416731.py:4: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\ASUS\Desktop\Data Analysis\Pandas\datasets\Airbnb_Open_Data.csv")


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license,Unnamed: 26
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN,NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN,NaN


In [164]:
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  str    
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  str    
 4   host name                       102193 non-null  str    
 5   neighbourhood group             102570 non-null  str    
 6   neighbourhood                   102583 non-null  str    
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  str    
 10  country code                    102468 non-null  str    
 11  instant_bookable                102494 non-null  object 
 12  cancellation_policy        

### Normalise the column names first

Real datasets have inconsistent column names - spaces, capitals, trailing whitespace.
Clean these *before* anything else so the rest of the notebook is predictable.

In [165]:
df
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ","_",regex=False)
)
df.columns.tolist()

['id',
 'name',
 'host_id',
 'host_identity_verified',
 'host_name',
 'neighbourhood_group',
 'neighbourhood',
 'lat',
 'long',
 'country',
 'country_code',
 'instant_bookable',
 'cancellation_policy',
 'room_type',
 'construction_year',
 'price',
 'service_fee',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'review_rate_number',
 'calculated_host_listings_count',
 'availability_365',
 'house_rules',
 'license',
 'unnamed:_26']

---
## 7.1 Handling Missing Data

The three questions to ask for **every** column with nulls:
1. How much is missing?
2. Is it missing *randomly*, or does the missingness itself mean something?
3. Drop the rows, drop the column, or fill it?

In [166]:
df.isna().sum().sort_values(ascending=False)

unnamed:_26                       102599
license                           102597
house_rules                        52131
last_review                        15893
reviews_per_month                  15879
country                              532
availability_365                     448
minimum_nights                       409
host_name                            406
review_rate_number                   326
calculated_host_listings_count       319
host_identity_verified               289
service_fee                          273
name                                 250
price                                247
construction_year                    214
number_of_reviews                    183
country_code                         131
instant_bookable                     105
cancellation_policy                   76
neighbourhood_group                   29
neighbourhood                         16
long                                   8
lat                                    8
host_id         

In [167]:
missing_percentage = (df.isna().mean() * 100).sort_values(ascending=False)
missing_percentage
missing_percentage[missing_percentage > 0].round(2)

unnamed:_26                       100.00
license                           100.00
house_rules                        50.81
last_review                        15.49
reviews_per_month                  15.48
country                             0.52
availability_365                    0.44
minimum_nights                      0.40
host_name                           0.40
review_rate_number                  0.32
calculated_host_listings_count      0.31
host_identity_verified              0.28
service_fee                         0.27
name                                0.24
price                               0.24
construction_year                   0.21
number_of_reviews                   0.18
country_code                        0.13
instant_bookable                    0.10
cancellation_policy                 0.07
neighbourhood_group                 0.03
neighbourhood                       0.02
long                                0.01
lat                                 0.01
dtype: float64

### Dropping

Rules of thumb:
- Column >50% missing and not critical -> drop the column.
- A handful of rows missing a **key** field (price, id) -> drop those rows.
- Everything else -> fill.

In [168]:
print("Original rows:",len(df))
print("Remaining after dropping Any row containing NULL:",len(df.dropna(how="any")))
print("Remaining after dropping only those rows containing All NULL:",len(df.dropna(how="all")))


Original rows: 102599
Remaining after dropping Any row containing NULL: 0
Remaining after dropping only those rows containing All NULL: 102599


In [169]:
cols_to_drop = missing_percentage[missing_percentage > 50].index.tolist()
cols_to_drop
print("dropping",cols_to_drop)

df = df.drop(columns=cols_to_drop,errors = "ignore")

"license" in df #fasle
"house_rules" in df #false
print(df.shape)
df = df.dropna(thresh=(len(df.columns) * 0.5))
df.shape

dropping ['unnamed:_26', 'license', 'house_rules']
(102599, 24)


(102599, 24)

### Filling

Pick the fill strategy per column *type*, not globally:
- numeric -> median (robust to the outliers we know are in here)
- categorical / text -> `"Unknown"` or the mode
- counts that mean "none happened" -> `0`

In [170]:
nums_columns = []
cat_columns = []

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        nums_columns.append(col)
    else:
        cat_columns.append(col)

#OR

nums_columns.clear()
cat_columns.clear()

nums_columns = df.select_dtypes(include="number").columns.tolist()
cat_columns = df.select_dtypes(exclude="number").columns.tolist()

print("Numeric dtype:",nums_columns)
print("Text dtype:", cat_columns)

Numeric dtype: ['id', 'host_id', 'lat', 'long', 'construction_year', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']
Text dtype: ['name', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'price', 'service_fee', 'last_review']


In [171]:
#numeric -> median ?? why cause its a prediction of how the data is varying you can say it CAN be but not approx
for col in nums_columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

In [172]:
for col in ["number_of_reviews", "reviews_per_month"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [173]:
#for categorical->unknown

for col in cat_columns:
    if df[col].isna().any():
        df[col] = df[col].fillna("Unknown")

In [174]:
#verify now
df.isna().sum() #always run the before cell connected to this

id                                0
name                              0
host_id                           0
host_identity_verified            0
host_name                         0
neighbourhood_group               0
neighbourhood                     0
lat                               0
long                              0
country                           0
country_code                      0
instant_bookable                  0
cancellation_policy               0
room_type                         0
construction_year                 0
price                             0
service_fee                       0
minimum_nights                    0
number_of_reviews                 0
last_review                       0
reviews_per_month                 0
review_rate_number                0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

---
## 7.2 Data Transformation

### Removing duplicates

In [175]:
df.duplicated().sum()
df[df.duplicated(keep=False)].sort_values(df.columns[0]).head(10) #default df.duplicated() only shows identical rows(merged) but keep=False will show both of them(split)

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
9098,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,United States,...,2008.0,$105,$21,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0
102474,6026161,Upper East Side 2 bedroom- close to Hospitals-,65193709566,verified,Juliana,Manhattan,Upper East Side,40.76222,-73.96030,United States,...,2008.0,$105,$21,30.0,2.0,6/8/2019,0.21,3.0,34.0,157.0
102475,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,United States,...,2008.0,$285,$57,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0
9099,6026714,Close to East Side Hospitals- Modern 2 Bedroom...,31072202372,verified,Juliana,Manhattan,Upper East Side,40.76249,-73.96217,United States,...,2008.0,$285,$57,30.0,6.0,1/31/2019,0.14,3.0,34.0,67.0
9100,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.76021,-73.96157,United States,...,2014.0,$586,$117,30.0,10.0,11/18/2018,0.22,5.0,34.0,211.0
102476,6027266,ACADIA Spacious 2 Bedroom Apt - Close to Hospi...,95854111798,verified,Juliana,Manhattan,Upper East Side,40.76021,-73.96157,United States,...,2014.0,$586,$117,30.0,10.0,11/18/2018,0.22,5.0,34.0,211.0
102477,6027818,*ENCHANTMENT* Upper East Side 2 bedroom- Sunny!,73401481508,unconfirmed,Juliana,Manhattan,Upper East Side,40.76244,-73.96031,United States,...,2006.0,$539,$108,30.0,9.0,9/30/2018,0.20,5.0,34.0,411.0
9101,6027818,*ENCHANTMENT* Upper East Side 2 bedroom- Sunny!,73401481508,unconfirmed,Juliana,Manhattan,Upper East Side,40.76244,-73.96031,United States,...,2006.0,$539,$108,30.0,9.0,9/30/2018,0.20,5.0,34.0,411.0
9102,6028371,*JAMES* Amazing Spacious 2 Bedroom- Bright!,37678424985,verified,Juliana,Manhattan,Upper East Side,40.76035,-73.96133,United States,...,2021.0,$806,$161,30.0,8.0,6/11/2019,0.27,4.0,34.0,411.0
102478,6028371,*JAMES* Amazing Spacious 2 Bedroom- Bright!,37678424985,verified,Juliana,Manhattan,Upper East Side,40.76035,-73.96133,United States,...,2021.0,$806,$161,30.0,8.0,6/11/2019,0.27,4.0,34.0,411.0


In [176]:
df = df.drop_duplicates()
df.sort_values(df.columns[0],ascending=True)
df.shape

(102058, 24)

In [177]:
id_col = "id" if "id" in df.columns else df.columns[0]
print("duplicate ids:", df[id_col].duplicated().sum())
df = df.drop_duplicates(id_col,keep="first")
df.shape

duplicate ids: 0


(102058, 24)

### Replacing sentinel values

Placeholder strings like `"N/A"` or `"-"` are *disguised* nulls. Convert them to real
NaN **before** you canonicalise categories, or they survive as fake category values.

In [178]:
df = df.replace(["N/A", "n/a", "NA", "-", "", "null", "nan", "Nan", "None"], np.nan)

for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Unknown")
df.isna().sum().sum()

np.int64(0)

### Fixing inconsistent categories

This is the classic `"USA" / "United States" / "U.S."` problem. In this dataset it
shows up in `neighbourhood_group` (Brooklyn misspelled, Manhattan abbreviated).

In [179]:
df
col = "neighbourhood_group"
df[col].value_counts(dropna=False)
fixes = {
    # "brookln": "Brooklyn",
    # "brooklyn": "Brooklyn",
    # "manhatan": "Manhattan",
    # "manhattan": "Manhattan", we already did this in the str.strip().str.title()
    "Brookln": "Brooklyn",
    "Manhatan": "Manhattan",
}

df[col] = df[col].str.strip().str.title()
df[col] = df[col].replace(fixes)
df[col].value_counts(dropna=False)

neighbourhood_group
Manhattan        43558
Brooklyn         41631
Queens           13197
Bronx             2694
Staten Island      949
Unknown             29
Name: count, dtype: int64

### Cleaning currency columns

`price` and `service_fee` arrive as strings like `"$1,200 "`. They must become numbers
before any outlier work or modelling.

In [180]:
df.columns.tolist()

for col in ["price","service_fee"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[$,]","",regex=True).str.strip(),errors="coerce")

df[col].describe()


count    101785.000000
mean        125.039249
std          66.325905
min          10.000000
25%          68.000000
50%         125.000000
75%         183.000000
max         240.000000
Name: service_fee, dtype: float64

In [181]:
#coerce might create new nan's 
df[col].isna()
for col in ["price","service_fee"]:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

### Renaming axis indexes

In [182]:
df = df.rename(columns={
    "neighbourhood_group": "borough",
        "calculated_host_listings_count": "host_listings",
        "availability_365": "availability"
})
df.columns.tolist()

['id',
 'name',
 'host_id',
 'host_identity_verified',
 'host_name',
 'borough',
 'neighbourhood',
 'lat',
 'long',
 'country',
 'country_code',
 'instant_bookable',
 'cancellation_policy',
 'room_type',
 'construction_year',
 'price',
 'service_fee',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'review_rate_number',
 'host_listings',
 'availability']

### Detecting and filtering outliers

Two approaches - use both and compare:
1. **Domain rules**: a price of \$0 or 1000 minimum nights is impossible.
2. **IQR rule**: statistically flag anything far outside the middle 50%.

In [183]:
print("price <=0:",(df["price"] <= 0).sum())
if "minimum_nights" in df.columns:
    print("minimum_nights > 365:", (df["minimum_nights"] > 365).sum())
    print("minimum_nights <= 0 :", (df["minimum_nights"] <= 0).sum())
if "availability" in df.columns:
    print("availability < 0    :", (df["availability"] < 0).sum())
    print("availability > 365  :", (df["availability"] > 365).sum())

price <=0: 0
minimum_nights > 365: 35
minimum_nights <= 0 : 13
availability < 0    : 431
availability > 365  : 2754


In [184]:
df = df[df["price"] > 0]

if "minimum_nights" in df.columns:
    df = df[(df["minimum_nights"] > 0) & (df["minimum_nights"] <= 365)]

if "availability" in df.columns:
    df = df[(df["availability"] >= 0) & (df["availability"] <= 365)]

In [185]:
def iqr_bounds(s,k=1.5):
    q1,q3 =s.quantile(.25),s.quantile(.75)
    IQR = q3-q1
    return q1 - (k * IQR), q3 + (k * IQR)

lo, hi = iqr_bounds(df["price"])
print(f"price IQR bounds: {lo:.2f} to {hi:.2f}")
print("outliers flagged:", ((df["price"] < lo) | (df["price"] > hi)).sum())

price IQR bounds: -517.00 to 1771.00
outliers flagged: 0


In [186]:
df["price_capped"] = df["price"].clip(lower=lo, upper=hi)
df[["price","price_capped"]].describe()

,price,price_capped
count,98826.000000,98826.000000
mean,625.757260,625.757260
std,331.276937,331.276937
min,50.000000,50.000000
25%,341.000000,341.000000
50%,625.000000,625.000000
75%,913.000000,913.000000
max,1200.000000,1200.000000


### Discretization and binning

In [187]:
binsss = [0, 100, 300, 600, 1000, np.inf]
labelsss = ["budget", "mid", "high", "premium", "luxury"]
df["price_band"] = pd.cut(df["price"],bins=binsss,labels=labelsss)
df["price_band"].value_counts().sort_index()


price_band
budget      4296
mid        17049
high       25773
premium    34537
luxury     17171
Name: count, dtype: int64

In [188]:
# qcut = equal-frequency bins (each bucket has ~the same number of listings)
df["price_quartile"] = pd.qcut(df["price"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
df["price_quartile"].value_counts().sort_index()

price_quartile
Q1    24756
Q2    24779
Q3    24647
Q4    24644
Name: count, dtype: int64

### Permutation and random sampling

In [189]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True) #note random state can be anything and it will be same if you use the same number 42 everytie
df_shuffled.head(3)#note random_state=None will give you a brand new order everytime

,id,name,host_id,host_identity_verified,host_name,borough,neighbourhood,lat,long,country,...,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,host_listings,availability,price_capped,price_band,price_quartile
0,14320051,Amazing Studio at the Time Square/52C,22721901471,verified,Irina,Manhattan,Hell's Kitchen,40.76174,-73.99228,United States,...,30.0,5.0,2/5/2019,0.22,4.0,20.0,333.0,1140.0,luxury,Q4
1,2815089,950 SQ FT ONE BEDROOM + 2 TERRACES,75859163955,verified,Courtney,Manhattan,Upper East Side,40.77924,-73.95405,United States,...,1.0,6.0,10/26/2014,0.10,2.0,1.0,291.0,1136.0,luxury,Q4
2,15092718,Charming Studio in Upper East Side,66870583929,verified,Juan Ignacio,Manhattan,Upper East Side,40.77265,-73.94833,United States,...,2.0,6.0,4/15/2018,0.27,3.0,1.0,0.0,118.0,mid,Q1


In [190]:
sampless = df.sample(n=(min(1000,len(df))), random_state=42)
sampless.shape

(1000, 27)

### Computing indicator / dummy variables

In [191]:
dummies = pd.get_dummies(df["borough"],prefix="borough")
dummies.head()

,borough_Bronx,borough_Brooklyn,borough_Manhattan,borough_Queens,borough_Staten Island,borough_Unknown
0,False,True,False,False,False,False
1,False,False,True,False,False,False
2,False,False,True,False,False,False
3,False,True,False,False,False,False
4,False,False,True,False,False,False


In [192]:
# one-hot the low-cardinality categoricals and join back
cat_for_dummies = [c for c in ["borough", "room_type", "price_band"] if c in df.columns]
df_encoded = pd.get_dummies(df, columns=cat_for_dummies, drop_first=True)
df_encoded.shape

(98826, 36)

---
## 7.3 Extension Data Types

Nullable integer and string dtypes let a column hold `NA` without silently becoming
`float64` or `object`.

In [193]:
df.dtypes

id                           int64
name                           str
host_id                      int64
host_identity_verified         str
host_name                      str
borough                        str
neighbourhood                  str
lat                        float64
long                       float64
country                        str
country_code                   str
instant_bookable            object
cancellation_policy            str
room_type                      str
construction_year          float64
price                      float64
service_fee                float64
minimum_nights             float64
number_of_reviews          float64
last_review                    str
reviews_per_month          float64
review_rate_number         float64
host_listings              float64
availability               float64
price_capped               float64
price_band                category
price_quartile            category
dtype: object

In [194]:
# nullable integer instead of float64-with-NaN
for col in ["number_of_reviews", "availability", "host_listings"]:
    if col in df.columns:
        df[col] = df[col].astype("Int64")

# proper string dtype instead of generic object
for col in ["name", "host_name"]:
    if col in df.columns:
        df[col] = df[col].astype("string")

df.dtypes

id                           int64
name                        string
host_id                      int64
host_identity_verified         str
host_name                   string
borough                        str
neighbourhood                  str
lat                        float64
long                       float64
country                        str
country_code                   str
instant_bookable            object
cancellation_policy            str
room_type                      str
construction_year          float64
price                      float64
service_fee                float64
minimum_nights             float64
number_of_reviews            Int64
last_review                    str
reviews_per_month          float64
review_rate_number         float64
host_listings                Int64
availability                 Int64
price_capped               float64
price_band                category
price_quartile            category
dtype: object

---
## 7.4 String Manipulation

The `.str` accessor plus regex. This is where most real cleaning time goes.

In [195]:
text_col = "name" if "name" in df.columns else cat_columns[0]
df[text_col].head(10)

0                   Clean & quiet apt home by the park
1                                Skylit Midtown Castle
2                  THE VILLAGE OF HARLEM....NEW YORK !
3                                              Unknown
4     Entire Apt: Spacious Studio/Loft by central park
6                                      BlissArtsSpace!
7                                      BlissArtsSpace!
8                      Large Furnished Room Near B'way
10                  Cute & Cozy Lower East Side 1 bdrm
11                    Beautiful 1br on Upper West Side
Name: name, dtype: string

In [203]:
# basic vectorised string methods
df[text_col + "_clean"] = (
    df[text_col].astype("string")
      .str.strip()
      .str.replace(r"\s+", " ", regex=True)     # collapse repeated whitespace
      .str.replace(r"[^\w\s]", "", regex=True)  # strip punctuation/emoji
      .str.title()
)
df[[text_col, text_col + "_clean"]].head(10)

,name,name_clean
0,Clean & quiet apt home by the park,Clean Quiet Apt Home By The Park
1,Skylit Midtown Castle,Skylit Midtown Castle
2,THE VILLAGE OF HARLEM....NEW YORK !,The Village Of Harlemnew York
3,Unknown,Unknown
4,Entire Apt: Spacious Studio/Loft by central park,Entire Apt Spacious Studioloft By Central Park
6,BlissArtsSpace!,Blissartsspace
7,BlissArtsSpace!,Blissartsspace
8,Large Furnished Room Near B'way,Large Furnished Room Near Bway
10,Cute & Cozy Lower East Side 1 bdrm,Cute Cozy Lower East Side 1 Bdrm
11,Beautiful 1br on Upper West Side,Beautiful 1Br On Upper West Side


In [204]:
# derived features from text
df["name_length"] = df[text_col].astype("string").str.len()
df["name_words"] = df[text_col].astype("string").str.split().str.len()
df[["name_length", "name_words"]].describe()

,name_length,name_words
count,98826.0,98826.000000
mean,37.634236,6.223555
std,10.815989,2.066988
min,1.0,1.000000
25%,31.0,5.000000
50%,39.0,6.000000
75%,47.0,8.000000
max,249.0,47.000000


### Rest Check From Airbnb_Chapter7_Cleaning.ipynb ###

In [205]:
df.to_csv("airbnb_cleaned.csv", index=False)
print("saved airbnb_cleaned.csv ->", df.shape)

saved airbnb_cleaned.csv -> (98826, 30)
